### Running the Surrogate Predictions
This code also generates error metrics for each of the trained models

In [2]:
# Run surrogate predictions
from gpPredict import *
import pandas as pd
import numpy as np
import json
import re

import matplotlib.pyplot as plt
import os

import pickle

from get_bw_params import *
from run_column_model import *

# Use latex for plots
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

In [3]:
# Load the calibration_info.csv file
calibration_info = pd.read_csv('gp_training_data/calibrations/calibration_info.csv')

# Open the data_split.csv files for both flexure and shear models
data_split_shear = pd.read_csv('gp_training_data/processed/gpModelShear/data_split.csv')
data_split_flexure = pd.read_csv('gp_training_data/processed/gpModelFlexure/data_split.csv')

# Merge the shear and flexure data splits
data_split = pd.concat([data_split_shear, data_split_flexure], ignore_index=True)
data_split.head()

,UniqueId,Name,Type,FailureType,ar,lrr,srr,alr,sdr,smr,...,sigp,rsmax,alpha,alpha1,alpha2,betam1,n,kappa_k,color,split
0,331,"Arakawa et al. 1987, No. 14",Spiral,Shear,1.090909,0.474999,0.069076,0.231296,0.972222,1.625200,...,0.229829,0.753973,0.000007,3.605178,0.073467,0.005424,2.321767,2.853569,red,0
1,332,"Arakawa et al. 1988, No. 15",Spiral,Flexure-Shear,1.636364,0.460800,0.032644,0.000000,2.083333,1.617065,...,2.302849,0.991142,0.000142,2.370283,0.865552,0.026052,3.754760,0.276921,green,0
2,328,"Arakawa et al. 1987, No. 11",Spiral,Shear,1.090909,0.518030,0.000000,0.252250,2.000000,3.585132,...,6.937252,0.784357,0.001759,0.730296,0.508968,0.046382,2.779707,1.614035,red,0
3,336,"Arakawa et al. 1988, No. 19",Spiral,Shear,1.636364,0.472615,0.033481,0.116019,2.083333,1.536306,...,4.240102,0.939044,0.000839,0.898241,0.045689,0.014780,3.288769,0.465055,red,0
4,280,"Ang et al. 1985, No. 14",Spiral,Flexure-Shear,2.000000,0.407644,0.022793,0.000000,1.666667,1.406019,...,0.335962,0.694097,0.000130,3.380845,0.467456,0.003175,2.635606,4.096478,green,0


In [4]:
# Create a new dataframe to store the error metrics
error_metrics = pd.DataFrame(columns=['UniqueId', 'split', 'err_cal_exp', 'err_surr_exp', 'err_surr_cal', 'ar', 'lrr', 'srr', 'alr', 'sdr', 'smr'])

#filter the calibration info to the following UniqueId
#sel_ids = [2, 5, 7, 22, 24, 49, 116, 137, 303, 228, 273, 281]
sel_ids = [299]

calibration_info = calibration_info[calibration_info['UniqueId'].isin(sel_ids)]
calibration_info.head()

,UniqueId,Name,Location,res_mean,res_median,res_std,res_max,res_min,gamma,kappa,...,lam,mup,sigp,rsmax,alpha,alpha1,alpha2,betam1,n,kappa_k
62,299,"Petrovski and Ristic 1984, M1E1",b25cf496-13f1-46ea-8d2a-0b8500e9ab78-007,0.091878,0.091682,0.001174,0.095454,0.089737,1.741679,1.05561,...,0.596109,0.327462,0.718706,0.717657,0.004369,3.529735,0.03528,0.006039,1.696971,0.816856


In [5]:
# Select the model to run the simulations
splits = ['split_00', 'split_01', 'split_02', 'split_03', 'split_04', 'split_05', 'no_split']
surr_path = 'gp_training_data/processed'
bw_params = ['gamma', 'kappa', 'eta1', 'sig', 'lam', 'mup', 'sigp', 'rsmax', 'alpha', 'alpha1', 'alpha2', 'betam1', 'n', 'kappa_k']
figures_path = 'Figures/surrogate_hysteresis'

# Iterate over all rows in calibration_info
#for ii in range(len(calibration_info)):
for split in splits:
    error_metrics = pd.DataFrame(columns=['UniqueId', 'split', 'err_cal_exp', 'err_surr_exp', 'err_surr_cal', 'ar', 'lrr', 'srr', 'alr', 'sdr', 'smr'])
    
    for ii in range(len(calibration_info)):
        # :::
        # Get the bouc-wen parameters
        # :::

        # Need to find whether it's in test or training dataset...
        UniqueId = calibration_info['UniqueId'].iloc[ii]

        # Find UniqueId in data_split
        row = data_split[data_split['UniqueId'] == UniqueId]
        
        # if row is empty, skip
        if row.empty:
            continue

        # For this row, get the calibrated bw model parameters
        calParams = calibration_info[bw_params].iloc[ii].values
        
        # Now, from data_split, get the nondimensional parameters
        ndParams = row[['ar', 'lrr', 'srr', 'alr', 'sdr', 'smr']].values[0]

        # Get surrogate-predicted bw model parameters 
        # ndParams, surrogate_path, split='no_split', mode='simple'
        predParams, predErr, failureMode = get_BW_params(ndParams, surr_path, split=split, mode='simple')
        
        # :::
        # Surrogate Performance Metrics
        # :::

        # Get the test_XXX.json file from test_data folder where XXX is the UniqueId
        with open('test_data/test_' + str(UniqueId).zfill(3) + '.json') as f:
            test_data = json.load(f)

        # Run the column model with the calParams
        calResults = run_model(test_data, calParams, do_plots=False)

        # Run the column model with the predParams
        surrResults = run_model(test_data, predParams, do_plots=False)

        print('Calibration Error: ', calResults['err_data']['mean_err'], ' --- std: ', calResults['err_data']['std_err'])
        print('Surrogate Error: ', surrResults['err_data']['mean_err'], ' --- std: ', surrResults['err_data']['std_err'])

        # Calculate difference between surrogate and calibration
        err_surr_cal = np.mean(np.abs(np.array(surrResults['sim_data']['nforce']) - np.array(calResults['sim_data']['nforce'])))
        std_surr_cal = np.std(np.abs(np.array(surrResults['sim_data']['nforce']) - np.array(calResults['sim_data']['nforce'])))

        print('Surrogate-Calibration Error: ', err_surr_cal, ' --- std: ', std_surr_cal)
        print('No Model Error: ', surrResults['err_data']['no_model_err'])
        print('1 - Model Error: ', surrResults['err_data']['one_model_err'])

        plt.figure(figsize=(3, 3))
        # Plot experimental data
        plt.plot(calResults['exp_data']['drift'], calResults['exp_data']['nforce'], 
                 label='Exp.', color='black', linestyle='-', linewidth=1.0, alpha=0.5)
        # Plot from calResults drift vs normalized force
        plt.plot(calResults['sim_data']['drift'], calResults['sim_data']['nforce'],
                 label='Cal.', color='red', linewidth=1.0, linestyle='-.', alpha=0.7)
        # Plot from surrResults displacement vs normalized force
        plt.plot(surrResults['sim_data']['drift'], surrResults['sim_data']['nforce'], 
                 label='GP', color='blue', marker='s', markevery=20, markersize=2.0,linewidth=1.5, linestyle='--', alpha=0.7)
        plt.xlabel('Drift Ratio $\Delta/h$')
        plt.ylabel('Normalized Shear $V/V_s$')
        plt.legend(loc='upper left', fontsize=6)

        # plt.title(test_data['Name']+'\n - PEER ID: ' + str(UniqueId))
        # Show only text up to the year number
        full_name = str(test_data.get('Name', ''))
        m = re.search(r'\b(\d{4})\b', full_name)
        if m:
            name_short = full_name[:m.end()].strip()
        else:
            name_short = full_name.split(',')[0].strip()
        title = name_short + '\n | PEER ID: ' + str(UniqueId)
        plt.title(title, fontsize=10)

        textstr = (
            f"Cal. MAE: {calResults['err_data']['mean_err']:.3f}\n"
            f"GP. MAE: {surrResults['err_data']['mean_err']:.3f}\n"
            f"GP/Cal MAE: {err_surr_cal:.3f}"
        )

        # Add a text box in the bottom right corner
        plt.gca().text(
            0.95, 0.05, textstr, transform=plt.gca().transAxes, fontsize=8,
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5)
        )
        plt.ylim([-1.05, 1.05])
        # Set yticks to be in increments of 0.2 from -1.2 to 1.2
        #plt.yticks(np.arange(-1.2, 1.3, 0.2))
        # Set xlim to be between min and max of the drift in the experimental data
        
        xlims = [np.floor(min(calResults['exp_data']['drift']) * 100) / 100, 
                np.ceil(max(calResults['exp_data']['drift']) * 100) / 100]
        # Get the limits from the experimental data for x-axis
        plt.xlim(xlims) # Set the x-limits based on the experimental data
        plt.tight_layout()
        
        # Save the figure to a file
        # Create directory if it doesn't exist for surrogate plots
        if not os.path.exists(os.path.join(figures_path, split)):
            os.makedirs(os.path.join(figures_path, split))

        plt.savefig(os.path.join(figures_path, split, f'UniqueId_{str(UniqueId).zfill(3)}.pdf'))
        print(f"Saved figure for UniqueId {UniqueId} in split {split} at {os.path.join(figures_path, split, f'UniqueId_{str(UniqueId).zfill(3)}.pdf')}")
        # Save as high-quality png (dpi=600)
        #out_fn = os.path.join(figures_path, split, f'UniqueId_{str(UniqueId).zfill(3)}.png')
        #plt.savefig(out_fn, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()

        # Append to error_metrics dataframe
        error_metrics = error_metrics.append(
            {
                'UniqueId': UniqueId, 
                'split': row['split'].values[0],
                'err_cal_exp': calResults['err_data']['mean_err'],
                'err_surr_exp': surrResults['err_data']['mean_err'],
                'no_model_err': surrResults['err_data']['no_model_err'],
                'one_model_err': surrResults['err_data']['one_model_err'],
                'err_surr_cal': err_surr_cal,
                'ar': ndParams[0],
                'lrr': ndParams[1],
                'srr': ndParams[2],
                'alr': ndParams[3],
                'sdr': ndParams[4],
                'smr': ndParams[5],
                'ColumnType': row['Type'].values[0],
                'FailureMode': row['FailureType'].values[0]
            }, ignore_index=True)

    # Save error_metrics to a csv file
    #if os.path.exists(os.path.join('gp_training_data', 'error_metrics', f'err_metrics_{split}.csv')):
    #    os.remove(os.path.join('gp_training_data', 'error_metrics', f'err_metrics_{split}.csv'))

    # error_metrics.to_csv(os.path.join('gp_training_data', 'error_metrics', f'err_metrics_{split}.csv'), index=False)

Flexure failure mode
Running flexure surrogate in path: gp_training_data/processed\gpModelFlexure\split_00\SimGpModel.json
Using input json: gp_training_data/processed\gpModelFlexure\scInput.json
Finished... Run Time =  3.080948829650879 sec
Finished... Run Time =  5.5602076053619385 sec
Calibration Error:  0.08957170867902814  --- std:  0.0588506694601833
Surrogate Error:  0.13357337980320788  --- std:  0.09375344080688783
Surrogate-Calibration Error:  0.0937318236387601  --- std:  0.0742000857056181
No Model Error:  0.42147238469179465
1 - Model Error:  0.5794992390040923
Saved figure for UniqueId 299 in split split_00 at Figures/surrogate_hysteresis\split_00\UniqueId_299.pdf
Flexure failure mode
Running flexure surrogate in path: gp_training_data/processed\gpModelFlexure\split_01\SimGpModel.json
Using input json: gp_training_data/processed\gpModelFlexure\scInput.json
Finished... Run Time =  3.0171849727630615 sec
Finished... Run Time =  3.5233073234558105 sec
Calibration Error:  0.0